# Figure 1: TCGA TMB and mutation prediction

Reads the TCGA run outputs from `Runs/TCGA/output/cpm/` and writes panels to `Figures/Figure1/output/`.

The plotting functions live in `figure1_helpers.py`; shared style constants live in `constants.py`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
FIGURES_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = FIGURES_DIR.parent
for path in (NOTEBOOK_DIR, FIGURES_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from constants import BEIGE, CMAP_BEIGE_PURPLE, CMAP_GOLD_PURPLE, CMAP_ORANGE_GREEN, GOLD, ORANGE, PURPLE, TEAL, make_linear_cmap
from figure1_helpers import (
    compute_tmb_purity_partial_correlations,
    load_mutation_prediction_summaries,
    compute_mutation_rank_sum_tests,
    load_tmb_results,
    mutation_cohort_summary,
    plot_fold_variability,
    plot_per_cancer_corr_p,
    plot_raw_tmb_by_cancer,
    plot_predicted_tmb_quartile_stratification,
    plot_roc_grid,
    plot_scatter_corr,
    plot_tmb_scatter,
    plot_tmb_scatter_by_cancer,
    dot_matrix_with_marginals,
    plot_auprc_prevalence_and_flag,
    plot_auprc_barplot,
    plot_ranked_gene_tiles,
    plot_mutation_rank_sum_supplement,
    save_table,
    summarize_mutation_rank_sum_tests,
)

TCGA_RUN = PROJECT_ROOT / "Runs" / "TCGA"
RESULTS_ROOT = TCGA_RUN / "output" / "cpm"
TMB_RESULTS_DIR = RESULTS_ROOT / "tmb"
MUTATION_LABEL_ROOT = TCGA_RUN / "data" / "mutations"
PURITY_PATH = PROJECT_ROOT / "data" / "TCGA_mastercalls.abs_tables_JSedit.fixed.txt"

FIGURE_ROOT = NOTEBOOK_DIR / "output"
SUPPLEMENT_ROOT = FIGURE_ROOT / "Supplemental"

CMAP_BEIGE_GREEN = make_linear_cmap([BEIGE, TEAL, PURPLE], "beige_teal_purple")
poster_cmap = make_linear_cmap([ORANGE, "#f2dac6", GOLD], "orange_peach_gold")

## Load TMB Results

In [ ]:
predictions_df, tmb_summary_df, tmb_fold_df, tmb_metadata = load_tmb_results(TMB_RESULTS_DIR)
print(f"TMB predictions: {predictions_df.shape[0]:,} samples x {predictions_df.shape[1]:,} columns")
print(f"Cancer types: {predictions_df['Cancer'].nunique():,}")
print(f"Expression features: {tmb_metadata.get('n_features', 'unknown')}")
print(f"Target: {tmb_metadata.get('target', 'unknown')}")
display(tmb_summary_df.head())
display(tmb_fold_df)

## Main Figure 1 TMB Panels

In [ ]:
# Figure 1b: pan-cancer held-out TMB prediction.
cancer_rho = plot_tmb_scatter(
    predictions_df,
    project_root=PROJECT_ROOT,
    figsize=(4, 3.3),
    cmap=CMAP_ORANGE_GREEN,
    save_path=FIGURE_ROOT / "figure1b_tmb_pan_cancer_scatter.pdf",
)

# Figure 1c: per-cancer TMB prediction performance.
tmb_per_cancer = plot_per_cancer_corr_p(
    predictions_df,
    project_root=PROJECT_ROOT,
    min_n=10,
    vmax_logp=50,
    figsize=(4.2, 2.9),
    orientation="vertical",
    cmap=CMAP_BEIGE_PURPLE,
    save_path=FIGURE_ROOT / "figure1c_tmb_per_cancer_spearman.pdf",
)
save_table(tmb_per_cancer, SUPPLEMENT_ROOT / "figure1c_tmb_per_cancer_spearman.csv", project_root=PROJECT_ROOT, index=False)

## TMB Supplemental Panels

In [ ]:
# Full scatter-by-cancer view.
plot_tmb_scatter_by_cancer(
    predictions_df,
    project_root=PROJECT_ROOT,
    n_rows=3,
    figsize=(9, 6),
    save_path=SUPPLEMENT_ROOT / "tmb_scatter_by_cancer_all.pdf",
)

# Raw TMB values by cancer: data-only scatter/bar plot.
plot_raw_tmb_by_cancer(
    predictions_df,
    project_root=PROJECT_ROOT,
    figsize=(7.0, 3.0),
    save_path=SUPPLEMENT_ROOT / "tmb_raw_by_cancer_scatter_barplot.pdf",
)

# Fold-level variability.
plot_fold_variability(
    tmb_fold_df,
    project_root=PROJECT_ROOT,
    figsize=(3.3, 2.5),
    save_path=SUPPLEMENT_ROOT / "tmb_fold_variability.pdf",
)

# ROC-style high-vs-low TMB summary using observed median TMB as the class label.
plot_roc_grid(
    predictions_df,
    project_root=PROJECT_ROOT,
    split_quantile=0.5,
    n_cols=7,
    figsize_per_panel=(1.25, 1.25),
    save_path=SUPPLEMENT_ROOT / "tmb_roc_grid_median_split.pdf",
)

# Stratification view: define top/bottom quartiles by predicted log-TMB and show observed TMB.
fig, ax, tmb_predicted_quartile_summary = plot_predicted_tmb_quartile_stratification(
    predictions_df,
    project_root=PROJECT_ROOT,
    quantile=0.25,
    pred_col="TMB_log2_pred",
    n_rows=2,
    figsize=(9, 4.4),
    yscale="log",
    save_path=SUPPLEMENT_ROOT / "tmb_predicted_quartile_observed_tmb_boxplots.pdf",
    table_save_path=SUPPLEMENT_ROOT / "tmb_predicted_quartile_observed_tmb_summary.csv",
)
display(tmb_predicted_quartile_summary.head())

# Correlation diagnostics.
plot_scatter_corr(
    x=tmb_per_cancer["mean_tmb"],
    y=tmb_per_cancer["spearman"],
    labels=tmb_per_cancer["Cancer"],
    xlabel="Mean TMB",
    ylabel="Spearman rho",
    color=PURPLE,
    project_root=PROJECT_ROOT,
    save_path=SUPPLEMENT_ROOT / "tmb_rho_vs_mean_tmb.pdf",
)
plot_scatter_corr(
    x=tmb_per_cancer["n"],
    y=tmb_per_cancer["spearman"],
    labels=tmb_per_cancer["Cancer"],
    xlabel="Sample size (n)",
    ylabel="Spearman rho",
    color=TEAL,
    project_root=PROJECT_ROOT,
    save_path=SUPPLEMENT_ROOT / "tmb_rho_vs_n.pdf",
)

## TMB-Purity Partial Correlation

Pooled and cancer-specific partial Spearman correlations between observed and out-of-fold predicted log2(coding TMB + 1), controlling for ABSOLUTE tumor purity. The pooled analysis is not adjusted for cancer type.

In [ ]:
PURITY_MIN_N = 10

tmb_purity_df, tmb_purity_partial_summary = compute_tmb_purity_partial_correlations(
    predictions_df,
    purity_path=PURITY_PATH,
    min_n=PURITY_MIN_N,
)
per_cancer_mask = tmb_purity_partial_summary["Cancer"].ne("__POOLED__")
save_table(
    tmb_purity_df,
    SUPPLEMENT_ROOT / "tmb_purity_partial_correlation_joined.csv",
    project_root=PROJECT_ROOT,
    index=False,
)
save_table(
    tmb_purity_partial_summary,
    SUPPLEMENT_ROOT / "tmb_purity_partial_correlation_summary.csv",
    project_root=PROJECT_ROOT,
    index=False,
)

print(
    f"Purity matched for {len(tmb_purity_df):,}/{len(predictions_df):,} TMB samples "
    f"across {tmb_purity_df['Cancer'].nunique()} cancers."
)
display(tmb_purity_partial_summary.head(1))
display(
    tmb_purity_partial_summary.loc[per_cancer_mask]
    .sort_values("purity_adjusted_spearman", ascending=False)
    .reset_index(drop=True)
)

## Mutation-Prediction Results

The main panels focus on pan-cancer target performance and recurrently predictable genes; supplemental outputs keep the broader cohort-level QC and prevalence-sensitive views.

In [ ]:
mutation_root = RESULTS_ROOT

cancer_dict, auprc_df, prevalence_df, normalized_auprc_df = load_mutation_prediction_summaries(
    [mutation_root],
    project_root=PROJECT_ROOT,
)

all_key = "all" if "all" in cancer_dict else ("All" if "All" in cancer_dict else None)
cancer_cols = [c for c in normalized_auprc_df.columns if c.lower() != "all"]

mutation_summary_df = mutation_cohort_summary(cancer_dict, mutation_root)
save_table(mutation_summary_df, SUPPLEMENT_ROOT / "mutation_cohort_qc_summary.csv", project_root=PROJECT_ROOT, index=False)

print(f"Mutation result root: {mutation_root.relative_to(PROJECT_ROOT)}")
print(f"Loaded mutation summaries for {len(cancer_dict)} cohorts ({len(cancer_cols)} cancer-specific cohorts + all cohort).")
print(f"Mutation targets in all cohort: {cancer_dict[all_key].shape[0] if all_key else 'missing'}")
display(mutation_summary_df.head())

In [ ]:
gene_tests = compute_mutation_rank_sum_tests(
    mutation_root=mutation_root,
    mutation_label_dir=MUTATION_LABEL_ROOT,
    min_mutated=3,
    min_wildtype=3,
)
rank_sum_summary = summarize_mutation_rank_sum_tests(gene_tests)

save_table(gene_tests, SUPPLEMENT_ROOT / "mutation_gene_rank_sum_tests.csv", project_root=PROJECT_ROOT, index=False)
save_table(rank_sum_summary, SUPPLEMENT_ROOT / "mutation_rank_sum_summary_by_cancer.csv", project_root=PROJECT_ROOT, index=False)
display(rank_sum_summary)

In [ ]:
fig, ax, rank_sum_plot_df = plot_mutation_rank_sum_supplement(
    rank_sum_summary,
    project_root=PROJECT_ROOT,
    figsize=(7.2, 3.0),
    save_path=SUPPLEMENT_ROOT / "mutation_rank_sum_summary_by_cancer.pdf",
    min_dot_size=28,
    max_dot_size=310,
)
display(rank_sum_plot_df.head())

## Main Figure 1 Mutation Panels

In [ ]:
# Figure 1e: top-ranked genes per cancer by normalized AUPRC.
fig, ax, top_scores, top_genes = plot_ranked_gene_tiles(
    auprc_df=normalized_auprc_df[cancer_cols],
    project_root=PROJECT_ROOT,
    top_n=10,
    figsize=(6, 3.2),
    cmap=CMAP_GOLD_PURPLE,
    text_contrast_threshold=0.68,
    xlabel="Rank within cancer (by normalized AUPRC)",
    cbar_label="Normalized AUPRC",
    save_path=FIGURE_ROOT / "figure1e_ranked_gene_tiles_nauprc.pdf",
)
save_table(top_scores, FIGURE_ROOT / "figure1e_ranked_gene_tiles_nauprc_scores.csv", project_root=PROJECT_ROOT)
save_table(top_genes, FIGURE_ROOT / "figure1e_ranked_gene_tiles_nauprc_genes.csv", project_root=PROJECT_ROOT)

# Figure 1d: pan-cancer gene AUPRC, ordered by prevalence.
auprc_barplot_color_by = "auprc"  # Choose "prevalence" or "auprc".
if all_key is not None:
    all_cancer_df = cancer_dict[all_key][["auprc", "prevalence"]]
    all_bar_df = plot_auprc_barplot(
        all_cancer_df.sort_values("prevalence", ascending=False).head(70),
        project_root=PROJECT_ROOT,
        figsize=(2.6, 6.5),
        cmap=CMAP_BEIGE_GREEN,
        color_by=auprc_barplot_color_by,
        save_path=FIGURE_ROOT / "figure1d_all_cohort_gene_auprc_barplot.pdf",
    )
    save_table(all_bar_df, FIGURE_ROOT / "figure1d_all_cohort_gene_auprc_barplot.csv", project_root=PROJECT_ROOT)

## Mutation Supplemental Panels

In [ ]:
# Recurrent predictable mutation genes across cancers, retained as supplemental.
mem, gene_counts, cancer_counts, recurrent_pts = dot_matrix_with_marginals(
    normalized_auprc_df[cancer_cols],
    project_root=PROJECT_ROOT,
    auprc_threshold=0.1,
    cmap="magma",
    top_bar_color="#555555",
    left_bar_color="#555555",
    cbar_label="Normalized AUPRC",
    top_n=20,
    figsize=(7.5, 4.2),
    save_path=SUPPLEMENT_ROOT / "mutation_per_cancer_gene_intersection_nauprc.pdf",
)
save_table(
    recurrent_pts,
    SUPPLEMENT_ROOT / "mutation_per_cancer_gene_intersection_nauprc_points.csv",
    project_root=PROJECT_ROOT,
    index=False,
)

# Raw AUPRC heatmap with prevalence dots, retained as supplemental because it is prevalence-sensitive.
fig, ax, raw_scores, raw_genes = plot_ranked_gene_tiles(
    auprc_df=auprc_df[cancer_cols],
    project_root=PROJECT_ROOT,
    prevalence_df=prevalence_df[cancer_cols],
    top_n=10,
    figsize=(6.5, 4),
    text_contrast_threshold=0.68,
    cmap=CMAP_GOLD_PURPLE,
    xlabel="Rank within cancer (by AUPRC)",
    cbar_label="AUPRC",
    save_path=SUPPLEMENT_ROOT / "mutation_ranked_gene_tiles_raw_auprc.pdf",
)
save_table(raw_scores, SUPPLEMENT_ROOT / "mutation_ranked_gene_tiles_raw_auprc_scores.csv", project_root=PROJECT_ROOT)
save_table(raw_genes, SUPPLEMENT_ROOT / "mutation_ranked_gene_tiles_raw_auprc_genes.csv", project_root=PROJECT_ROOT)

# Per-cancer AUPRC-vs-prevalence flag plots (Figure 1f uses ESCA, HNSC, LIHC, SKCM).
flag_dir = SUPPLEMENT_ROOT / "mutation_auprc_vs_prev_flag"
flagged_tables = []
for cancer, df in cancer_dict.items():
    if df.shape[0] <= 5:
        continue
    flagged = plot_auprc_prevalence_and_flag(
        df,
        cancer=cancer,
        min_points=15,
        project_root=PROJECT_ROOT,
        cmap=poster_cmap,
        figsize=(3.3, 2),
        save_path=flag_dir / f"{cancer}_auprc_vs_prev_flag.pdf",
    )
    if flagged is not None and not flagged.empty:
        flagged.insert(0, "cohort", cancer)
        flagged_tables.append(flagged)
if flagged_tables:
    mutation_flagged_genes = pd.concat(flagged_tables, ignore_index=True)
    save_table(mutation_flagged_genes, SUPPLEMENT_ROOT / "mutation_auprc_vs_prev_flagged_genes.csv", project_root=PROJECT_ROOT, index=False)

# Long table for downstream supplemental summaries.
long_rows = []
for cohort in cancer_cols:
    df = cancer_dict[cohort].copy()
    df.insert(0, "cohort", cohort)
    long_rows.append(df)
mutation_long_df = pd.concat(long_rows, ignore_index=True)
save_table(mutation_long_df, SUPPLEMENT_ROOT / "mutation_gene_metrics_long.csv", project_root=PROJECT_ROOT, index=False)

## Output Inventory

In [ ]:
for folder in [FIGURE_ROOT, SUPPLEMENT_ROOT]:
    print(folder.relative_to(PROJECT_ROOT))
    for path in sorted(folder.glob("*.pdf")):
        print(" -", path.name)